# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [ ]:
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !mkdir -p _tools
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_models.py -o _tools/colab_models.py
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_neural_trainer.py -o _tools/colab_neural_trainer.py
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

In [ ]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (H=60с, K=3.0, L=180с)
# После КАЖДОГО рана автоматически сохраняются и выгружаются в облако:
#   - лучшие веса model_win_00.keras + метаданные meta_win_00.json
#   - состояние ранов progress_win_00.json + лог training_log_h60s.txt
# При перезапуске или переносе на другой сервер обучение автоматически продолжится со следующего рана!
import os, shutil
FILE_ID = "1DVHqdiHBw2_2Iqpo6K0fvr3a49HqVDD_"              # Свежий датасет от 27.10 (H=5: 34, H=12: 11, H=25: 5, H=60: 1)
HORIZON = 60                                               # Горизонт H (сек): 60 (K=3.0, L=180с, Train=900k, Test=450k)
WINDOWS = "0"                                              # Строгое 100%-OOS окно win_00
ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"   # Все 4 архитектуры через запятую
RUNS_PER_ARCH = 10                                         # Число независимых ранов на архитектуру (синхронизировано с ночным циклом)
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 2e-3                                                  # Скорость обучения AdamW
SYNC_BUCKET = "prl_h60s_k3_ckpt"                           # Постоянный облачный бакет авто-сохранения после каждого рана
RESUME_URL = "https://filebin.net/prl_h60s_k3_ckpt/prl_ensemble_pack_h60s.zip"

# В Colab очищаем старый локальный кэш пакета, чтобы гарантированно подтянуть свежий ночной чекпоинт (r07/10)
if os.path.exists("/content"):
    shutil.rmtree(f"prl_ensemble_pack_h{HORIZON}s", ignore_errors=True)
    if os.path.exists(f"prl_ensemble_pack_h{HORIZON}s.zip"):
        os.remove(f"prl_ensemble_pack_h{HORIZON}s.zip")

df_leaderboard = run_colab_pipeline(
    file_id=FILE_ID,
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR,
    sync_bucket=SYNC_BUCKET, resume_url=RESUME_URL,
)


☁️ Авто-восстановление прогресса с предыдущего сервера: https://filebin.net/prl_h60s_k3_ckpt/prl_ensemble_pack_h60s.zip
☁️ Прямое скачивание prl_neural_colab_data.npz с Google Drive (ID=1DVHqdiHBw2_2Iqpo6K0fvr3a49HqVDD_)...
⚡ Подготовка memory-mapped кэша на диске (экономит ~4 ГБ RAM)...
⏩ [CONV1D+GRU | win_00] Чекпоинт уже готов -> IC: +0.1646 | AUC: 0.7061 (пропуск)
⏩ [CAUSAL_CNN | win_00] Чекпоинт уже готов -> IC: +0.0617 | AUC: 0.6345 (пропуск)

🚀 ОБУЧЕНИЕ ОКНА WIN_00 | H=60с (K=3.0, L=180с) | Train=900,000 (2026-10-10 10:17..2026-10-22 05:20)
🔄 [ATTENTION | win_00] Предзагружен лидер r03/10 (Loss: 1.9735 | IC: +0.0690 | AUC: 0.6672 | Top-2%: +0.07) — продолжаем с r09/10!
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.001000
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.000500
   ⏳ [Elastic EarlyStop] Достигнут эластичный потолок (6/10 эп.).
  [ATTENTION | win_00 | r09/10] ✅ OK (1107.6с) -> Loss: 1.9900 | IC: +0.0899 | AUC: 0

In [8]:
# 3. Выгрузка готового архива моделей (авто-упаковка при прерванной 2-й ячейке)
import os, shutil
HORIZON = globals().get("HORIZON", 60)
!pkill -9 -f "colab_neural_trainer" 2>/dev/null || true
PACK_DIR = f"prl_ensemble_pack_h{HORIZON}s"
ZIP_NAME = f"{PACK_DIR}.zip"
if os.path.exists(PACK_DIR):
    shutil.make_archive(PACK_DIR, "zip", root_dir=".", base_dir=PACK_DIR)
if os.path.exists(ZIP_NAME):
    print(f"📦 Архив найден: {ZIP_NAME} ({os.path.getsize(ZIP_NAME) / (1024*1024):.2f} MB)")
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://x0.at/ | head -c 200 && echo " (x0.at)"
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://store1.gofile.io/uploadFile | head -c 400 && echo " (gofile)"
    !curl -s --max-time 60 -H "Accept: application/json" -T "{ZIP_NAME}" https://filebin.net/prl_h{HORIZON}s_pack/{ZIP_NAME} | head -c 300 && echo " (filebin)"


^C
📦 Архив найден: prl_ensemble_pack_h25s.zip (4.12 MB)
https://x0.at/8mRD.zip
 (x0.at)
{"data":{"code":"qvnevKLa","createTime":1792827375,"downloadPage":"https://gofile.io/d/ap9BHslj","guestToken":"UGAqWj5apMxgbQ2eCPXOTcxgEZoTx2cd","id":"d596747f-0ce5-4834-8b0f-236ad84763c5","md5":"7ef1b9491a547c7a4347e4862a883812","mimetype":"application/zip","modTime":1792827375,"name":"prl_ensemble_pack_h25s.zip","parentFolder":"c3c7d3bd-574e-491d-aa22-1dddc597bd7d","parentFolderCode":"ap9BHslj", (gofile)
{
    "bin": {
        "id": "prl_h25s_pack",
        "readonly": false,
        "bytes": 4335814,
        "bytes_readable": "4.3 MB",
        "files": 1,
        "updated_at": "2026-10-24T07:36:27.356974Z",
        "updated_at_relative": "now",
        "created_at": "2026-10-21T12:17:32.137515Z",
  (filebin)
